In [0]:
from pyspark.sql.functions import current_timestamp, col, lit
import re

# =========================================================
# CONFIGURATION
# =========================================================

catalog = "company_risk_intelligence_platform"
schema = "bronze"

# =========================================================
# SOURCE CONFIGURATION
# =========================================================

source_configs = {

    "companies_house": {

        "base_path": "/Volumes/company_risk_intelligence_platform/bronze/raw_data/companies_house/",

        "tables": {
            "ch_filing_history": "filing_history",
            "ch_overview": "overview",
            "ch_people": "people"
        },

        "recursive_lookup": False
    },

    "yfinance": {

        "base_path": "/Volumes/company_risk_intelligence_platform/bronze/raw_data/yfinance/",

        "tables": {
            "yf_stock": "stock",
            "yf_news": "news",
            "yf_income_statement": "income_statement",
            "yf_balance_sheet": "balance_sheet",
            "yf_cashflow": "cashflow",
            "yf_info": "info"
        },

        "recursive_lookup": True
    }
}

# =========================================================
# CLEAN COLUMN NAMES
# =========================================================

def clean_column_names(df):

    cleaned_cols = []

    for c in df.columns:

        clean_name = (
            c.strip()
             .lower()
             .replace(" ", "_")
             .replace("-", "_")
        )

        clean_name = re.sub(r"[^a-zA-Z0-9_]", "", clean_name)

        cleaned_cols.append(clean_name)

    return df.toDF(*cleaned_cols)

# =========================================================
# ADD METADATA
# =========================================================

def add_metadata_columns(df, source_name):

    return (
        df.withColumn("ingestion_ts", current_timestamp())
          .withColumn("source_system", lit(source_name))
          .withColumn("file_path", col("_metadata.file_path"))
    )

# =========================================================
# MAIN BRONZE INGESTION LOOP
# =========================================================

for source_name, config in source_configs.items():

    print(f"\n========== Processing Source: {source_name} ==========")

    base_path = config["base_path"]
    table_mapping = config["tables"]
    recursive_lookup = config["recursive_lookup"]

    for table, folder in table_mapping.items():

        print(f"\nProcessing Table: {table}")

        input_path = f"{base_path}{folder}/"

        try:

            # ---------------------------------------------
            # READ JSON
            # ---------------------------------------------

            reader = (
                spark.read
                     .option("multiline", "true")
            )

            if recursive_lookup:

                reader = reader.option("recursiveFileLookup", "true")

            df = reader.json(input_path)

            # ---------------------------------------------
            # CLEAN COLUMNS
            # ---------------------------------------------

            df = clean_column_names(df)

            # ---------------------------------------------
            # ADD METADATA
            # ---------------------------------------------

            df = add_metadata_columns(df, source_name)

            # ---------------------------------------------
            # WRITE DELTA TABLE
            # ---------------------------------------------

            (
                df.write.format("delta")
                  .mode("overwrite")
                  .option("overwriteSchema", "true")
                  .saveAsTable(f"{catalog}.{schema}.{table}")
            )

            print(f"Loaded Table: {table}")

        except Exception as e:

            print(f"Skipping {table}")
            print(f"Reason: {e}")

print("\n========== ALL BRONZE TABLES PROCESSED ==========")